# UC-ASR — Kaggle GPU Run

Runs the full UC-ASR pipeline at the **`kaggle`** scale (real HuggingFace Qwen2.5-3B 4-bit on a T4).

## Before you run
1. In the Kaggle notebook sidebar, open **Settings** and set **Accelerator = GPU T4**.
2. In the same panel turn **Internet = On** (needed to download the model + dataset).
3. Expect roughly **8–11 hours** for the default `kaggle` cap (300 eval users on seed 42).
4. Everything runs from `/kaggle/working`, so `cache/` (SQLite LLM cache), `results/`, and `data/`
   persist in the session output. Reruns resume cheaply from the SQLite cache — raise
   `evaluation.kaggle_max_eval_users` across sessions to score more users without recomputing.


In [ ]:
# Install dependencies. bitsandbytes is gated OFF on Windows in requirements.txt
# (CUDA-only), so on Kaggle/Linux we install it explicitly for 4-bit loading.
!pip install -q -r requirements.txt bitsandbytes

In [ ]:
# Confirm the GPU is visible before launching the (long) run.
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

In [ ]:
# Get the repo into /kaggle/working so all relative paths (cache/, results/, data/) are writable
# and persist. Pick ONE option.

# Option A — clone from git (needs Internet = On):
import os
os.chdir('/kaggle/working')
!git clone https://github.com/ArunMuthukumarasamy/UC_ASR_Thesis.git
os.chdir('/kaggle/working/UC_ASR_Thesis')

# Option B — copy from an uploaded Kaggle dataset at /kaggle/input into /kaggle/working
# (replace <dataset-slug> with the folder name under /kaggle/input):
# import shutil, os
# shutil.copytree('/kaggle/input/<dataset-slug>', '/kaggle/working/UC_ASR_Thesis', dirs_exist_ok=True)
# os.chdir('/kaggle/working/UC_ASR_Thesis')

print('cwd:', os.getcwd())

In [ ]:
# Launch the full pipeline at kaggle scale (~8–11h for 300 eval users, seed 42).
!python run_pipeline.py --scale kaggle --stage all

In [ ]:
# Inspect outputs.
import os
print('results/ contents:')
for name in sorted(os.listdir('results')):
    print(' ', name)
print()
with open('results/thesis_main_table.md', encoding='utf-8') as fh:
    print(fh.read())